# Vision: convolutions and borrowed weights

> Why a dense layer is the wrong tool for pixels, what a convolution actually assumes, and the single technique that makes computer vision practical on 200 images.

Read this chapter at `/learn/11-vision-and-transfer/`. Exported from `src/content/chapters/11-vision-and-transfer.mdx` — edit there, not here.


An image is just numbers, so a dense network can happily eat one.

It shouldn't. And the reason is worth understanding *precisely*, because the same
line of reasoning produces every remaining architecture in this book. Get it once
here and chapters 12, 13 and 14 will feel like variations rather than surprises.

## Why not just flatten it?

In [ ]:
import numpy as np, matplotlib.pyplot as plt
from sklearn.datasets import load_digits

digits = load_digits()
images = digits.images          # (1797, 8, 8)
print("tiny images:", images.shape)

for h, w, hidden in [(8, 8, 100), (28, 28, 100), (224, 224, 1000)]:
    n_in = h * w * (3 if h > 100 else 1)
    print(f"{h}x{w} -> dense({hidden})   {n_in * hidden:,} weights in the first layer alone")

A 224×224 colour photograph into a modest first layer is **150 million weights**.
For one layer. Of one model. And you haven't learned anything yet.

But the cost isn't even the worst part. Think about what those weights *are*.

Each one is a separate parameter for one specific pixel position. So a network
that learns "there's a vertical edge here" at pixel (30, 40) has learned
absolutely nothing about a vertical edge at (31, 40). One pixel over. Every
position has to be learned independently, from its own separate examples.

That's not a network that's expensive. That's a network that's *badly designed for
the job*.

Both problems have exactly one cause: a dense layer knows nothing about the
**structure** of its input.

Here's a test that makes it vivid. Shuffle all the pixels of every image — the
same shuffle each time — and a dense network performs identically. It literally
cannot tell.

For a spreadsheet that's correct behaviour, and in chapter 7 it's why trees do
well. For an image it throws away the single most useful fact you have: nearby
pixels are related, and a cat is a cat wherever it happens to be standing.

## A convolution, by hand

A convolution slides a small grid of weights across the
image and records the dot product at every position.

The important word is *same*. The **same** weights, at every position.

In [ ]:
def convolve2d(image, kernel):
    kh, kw = kernel.shape
    h, w = image.shape
    out = np.zeros((h - kh + 1, w - kw + 1))
    for i in range(out.shape[0]):
        for j in range(out.shape[1]):
            out[i, j] = (image[i:i + kh, j:j + kw] * kernel).sum()
    return out

vertical   = np.array([[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]], float)   # Sobel
horizontal = vertical.T
blur       = np.ones((3, 3)) / 9

img = images[0]
fig, ax = plt.subplots(1, 4, figsize=(9, 2.5))
for a, (name, k) in zip(ax, [("original", None), ("vertical edges", vertical),
                             ("horizontal edges", horizontal), ("blur", blur)]):
    a.imshow(img if k is None else convolve2d(img, k), cmap="gray")
    a.set_title(name, fontsize=9); a.axis("off")
plt.tight_layout()

**Nine numbers** turned an image into an edge map.

Change the nine numbers and you get a completely different detector — blur,
sharpen, horizontal edges, whatever you like. And here's the thing: *learning the
nine numbers* is exactly, precisely what a convolutional layer does. Nothing more.

In [ ]:
h = w = 224
dense_first_layer = (h * w * 3) * 64
conv_first_layer  = 3 * 3 * 3 * 64 + 64          # 3x3 kernel, 3 in, 64 out
print(f"dense(64) on 224x224x3 : {dense_first_layer:>12,} weights")
print(f"conv 3x3, 64 filters   : {conv_first_layer:>12,} weights")
print(f"ratio                  : {dense_first_layer / conv_first_layer:>12,.0f}x fewer")

Thousands of times fewer parameters — and every single one is reused at every
position.

That reuse *is* the idea, and it encodes a genuine claim about the world:

**What a thing looks like does not depend on where it is.**

It's the difference between a lookup table with an entry per position, and a
single function applied over a sliding window — `slice.windows(3).map(f)`, in two
dimensions.

You'd never write the lookup table, and for exactly the same reason: the
positions aren't independent, so giving each one its own parameters is both
wasteful *and* worse at generalising. You'd factor out the function. That's all a
convolution is.

That assumption is a **prior**, and priors can be wrong.

Convolutions assume translation equivariance and local structure. Both hold for
photographs. Neither holds for a spreadsheet — which is why CNNs never worked on
tabular data, and now you know it wasn't a tuning problem.

And both hold only *weakly* for a medical scan, where absolute position is often
diagnostic. Which is why medical imaging models frequently feed position back in
explicitly, undoing part of the very thing that made convolutions work.

Choosing an architecture is choosing an assumption. That one sentence is most of
what architecture research actually is.

## Channels, stride, padding, pooling

Four knobs, and they're the entire vocabulary of a convolutional layer. Learn
these four and you can read any CNN paper's architecture table.

**Channels.** A colour image has 3 input channels. A layer with 64 filters
produces 64 output channels — 64 different learned detectors, each looking at all
the input channels at once. So a `Conv2d(3, 64, 3)` weight has shape
`(64, 3, 3, 3)`.

**Stride.** How far the window jumps between positions. Stride 2 halves the
output size.

**Padding.** Zeros around the border so the output keeps its size. Without it,
every 3×3 convolution shrinks the image by 2 pixels — and thirty layers would
leave you with nothing at all.

**Pooling.** Downsample by taking the max (or mean) of each small block. Fewer
positions, a larger effective field of view, and a bit of robustness to small
shifts.

In [ ]:
def conv_out(size, kernel, stride=1, pad=0):
    return (size + 2 * pad - kernel) // stride + 1

size = 224
print(f"{'layer':28s} {'out':>6s}")
for label, k, s, p in [("conv 3x3 pad 1", 3, 1, 1), ("conv 3x3 pad 1", 3, 1, 1),
                       ("maxpool 2x2 stride 2", 2, 2, 0), ("conv 3x3 pad 1", 3, 1, 1),
                       ("maxpool 2x2 stride 2", 2, 2, 0)]:
    size = conv_out(size, k, s, p)
    print(f"{label:28s} {size:>6d}")

That shrinking ladder is the standard shape of every CNN: spatial resolution
falls, channel count rises.

And there's a nice way to read that trade. You're exchanging **"where"** for
**"what"**. Early layers know precisely where an edge is but not what it means.
Late layers know there's a face *somewhere* but have lost the exact pixel. Which
is usually what you wanted.

## A real CNN

In [ ]:
# needs PyTorch (this kernel has it; the browser runtime does not)
import torch, torch.nn as nn
from sklearn.model_selection import train_test_split
from sklearn.datasets import load_digits

d = load_digits()
X = torch.tensor(d.images, dtype=torch.float32).unsqueeze(1) / 16.0   # (n, 1, 8, 8)
y = torch.tensor(d.target, dtype=torch.long)
Xtr, Xva, ytr, yva = train_test_split(X, y, test_size=0.25, random_state=0, stratify=y)

class ConvNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 16, 3, padding=1), nn.BatchNorm2d(16), nn.ReLU(),
            nn.Conv2d(16, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(),
            nn.MaxPool2d(2),                       # 8x8 -> 4x4
        )
        self.head = nn.Sequential(nn.Flatten(), nn.Linear(32 * 4 * 4, 10))
    def forward(self, x):
        return self.head(self.features(x))

torch.manual_seed(0)
net = ConvNet()
print("parameters:", sum(p.numel() for p in net.parameters()))

In [ ]:
# needs PyTorch (this kernel has it; the browser runtime does not)
from torch.utils.data import TensorDataset, DataLoader

dl = DataLoader(TensorDataset(Xtr, ytr), batch_size=64, shuffle=True)
opt = torch.optim.AdamW(net.parameters(), lr=3e-3)
loss_fn = nn.CrossEntropyLoss()          # softmax is inside the loss

for epoch in range(8):
    net.train()
    for xb, yb in dl:
        opt.zero_grad(); loss_fn(net(xb), yb).backward(); opt.step()
    net.eval()
    with torch.no_grad():
        acc = (net(Xva).argmax(1) == yva).float().mean()
    print(f"epoch {epoch+1}  valid accuracy {acc:.3f}")

Same five lines as yesterday. Different architecture, identical loop — which is
rather the point.

Note `argmax(1)`: the model outputs ten scores and you take
the index of the largest. And `CrossEntropyLoss` takes **logits** and integer
class labels, applying softmax internally — the same
logits-not-probabilities rule from yesterday.

**BatchNorm** deserves a sentence, and an honest one. It normalises each channel's
activations to zero mean and unit variance across the batch, which keeps the
scale of activations stable through depth and lets you use a larger learning
rate.

It was introduced in 2015 with an explanation — "internal covariate shift" — that
later work largely disputes. So the current state of knowledge is: nobody is
entirely sure *why* it works so well, and everybody uses it anyway. I'd rather
tell you that than pretend the story is settled.

## Transfer learning

Now the technique that makes computer vision practical for people who don't own a
data centre. If you take one thing from today, take this.

A network trained on ImageNet has learned, in its early layers, edge detectors,
texture detectors and shape detectors. And here's the key observation: **none of
that is specific to the thousand ImageNet classes.** Edges are edges. Texture is
texture. A curve is a curve whether it belongs to a cat or a car or a chest X-ray.

So: take those weights. Throw away the final classification layer. Bolt on a new
one for *your* classes. Train.

In [ ]:
# needs PyTorch (this kernel has it; the browser runtime does not)
import torchvision
from torchvision.models import resnet18, ResNet18_Weights

weights = ResNet18_Weights.DEFAULT
backbone = resnet18(weights=weights)     # downloads ~45 MB once, then cached

total = sum(p.numel() for p in backbone.parameters())
print(f"resnet18: {total:,} parameters, pretrained on ImageNet")
print("final layer:", backbone.fc)

# Replace the 1000-class head with your own.
backbone.fc = nn.Linear(backbone.fc.in_features, 5)     # 5 classes of yours
print("replaced with:", backbone.fc)

# Freeze everything except the new head, for the first phase of training.
for name, p in backbone.named_parameters():
    p.requires_grad = name.startswith("fc")
trainable = sum(p.numel() for p in backbone.parameters() if p.requires_grad)
print(f"trainable now: {trainable:,} of {total:,}  ({trainable / total:.2%})")

This is the difference between needing a million labelled images and needing two
hundred.

That's not a marginal optimisation. It is the entire reason a person with a
laptop can build a working image classifier in an afternoon — and it is
comfortably the most valuable practical technique in this chapter.

Somebody else spent the GPU-years. You get the edges for free.

The standard recipe, worth memorising:

1. **Freeze the backbone, train the new head** for a few epochs. The head starts
   random, and a random head sends garbage gradients backwards through weights
   that took a small fortune to tune. Don't let it.
2. **Unfreeze everything, train at a much lower learning rate** — often 10 to 100
   times lower. You're *nudging* good weights, not searching from scratch.
3. Optionally use **discriminative learning rates**: lower for early layers
   (generic edges, already correct) and higher for late layers (task-specific,
   need to change).

In [ ]:
# needs PyTorch (this kernel has it; the browser runtime does not)
for p in backbone.parameters():
    p.requires_grad = True

opt = torch.optim.AdamW([
    {"params": backbone.layer1.parameters(), "lr": 1e-5},   # generic: barely touch
    {"params": backbone.layer4.parameters(), "lr": 1e-4},   # task-ish
    {"params": backbone.fc.parameters(),     "lr": 1e-3},   # brand new
])
print("three parameter groups, three learning rates:")
for g in opt.param_groups:
    print(f"  lr={g['lr']:<8} {sum(p.numel() for p in g['params']):>10,} params")

## Augmentation

Free data, made out of the data you already have.

In [ ]:
img = images[7]
fig, ax = plt.subplots(1, 5, figsize=(10, 2.2))
variants = [
    ("original",   img),
    ("flip",       img[:, ::-1]),
    ("shift",      np.roll(img, 1, axis=0)),
    ("brighter",   np.clip(img * 1.5, 0, 16)),
    ("noise",      np.clip(img + np.random.default_rng(0).normal(0, 1.2, img.shape), 0, 16)),
]
for a, (name, v) in zip(ax, variants):
    a.imshow(v, cmap="gray"); a.set_title(name, fontsize=9); a.axis("off")
plt.tight_layout()

Each variant is a **new training example with the same label** — and each one
teaches the model an invariance. A rotated cat is a cat. A darker cat is a cat. A
slightly noisy cat is still, reassuringly, a cat.

Ten thousand photos become effectively a hundred thousand, and you didn't label a
single one.

Augment with transformations that **preserve the label**, and think about it
properly rather than copying somebody's config.

Horizontal flips are fine for cats. They are catastrophic for handwritten digits,
because a mirrored 2 is not a 2. Also bad for road signs, text, and anything
where chirality carries meaning. You would be *teaching the model something
false*.

And: **augment the training set only.** Augmenting validation makes your metric
measure a different, easier problem, and the number quietly stops meaning
anything at all.

**"My convolution output is a weird size."** Use `conv_out` above. The commonest
cause is forgetting `padding=1` on a 3×3 conv, which silently shrinks by 2 each
layer.

**"`Expected 4-dimensional input`"** — Conv2d wants `(batch, channels, height,
width)`. A single image needs `.unsqueeze(0)` for batch and often
`.unsqueeze(1)` for a missing channel dimension. That's what the `/ 16.0` cell
above is doing.

**"Why divide the images by 16?"** The digits dataset stores pixel values 0–16.
Dividing puts them in [0, 1], which keeps activations in a sane range — the same
standardisation habit from chapter 5, wearing different clothes.

**"The pretrained model download failed with an SSL error."** That's the
certificate issue on the [Setup](/setup/) page. One double-click fixes it
permanently.

**"My transfer-learned model got *worse* after unfreezing."** Learning rate too
high for phase 2. You're supposed to be nudging. Try dividing by 10, then 100 —
this is the single most common transfer learning mistake.

## Where vision is now

CNNs dominated from 2012 to about 2020. Then the **Vision Transformer** (2020)
showed something startling: cut an image into 16×16 patches, treat them as a
sequence of tokens, and a plain transformer
([Chapter 13](/learn/13-attention-and-transformers/)) beats a CNN.

*Given enough data.*

And that caveat is the genuinely interesting part. A ViT has **no built-in
assumption** about locality or translation. It has to learn from data what a CNN
was handed for free.

With ImageNet-scale data it does learn it — and then goes further, because it
isn't constrained by an assumption that was only ever approximately true. With
5,000 images it loses badly, because it's spending its capacity rediscovering
edges.

That pattern is worth carrying out of this chapter, because it recurs everywhere.

**A structural prior is a loan against data.** It buys you performance when data
is scarce, and it costs you a ceiling when data is plentiful.

Watch the field make that trade over and over: convolutions gave way to
attention. Hand-crafted image features gave way to learned features. Hand-written
grammars gave way to language models. Each time, the structure that helped when
data was expensive became the constraint that capped you once data was cheap.

Which means the question "which architecture is best?" simply has no answer
without "at what data scale?" — and if somebody answers it without asking you
that, they've told you something about themselves rather than about
architectures.

Modern practice mostly uses hybrids (ConvNeXt, Swin) that keep a bit of the
locality prior and take attention's flexibility. Have your loan and repay it too.

In [ ]:
# 1. Design a 3x3 kernel that detects diagonal edges. Apply it with convolve2d.
#
# 2. Apply the vertical Sobel kernel twice in a row. What does the second
#    application respond to? (Think about what "edges of an edge map" means.)
#
# 3. Work out the output size for a 32x32 input through:
#    conv 5x5 no pad -> maxpool 2 -> conv 3x3 pad 1 -> maxpool 2.
#    Then verify with conv_out.

print("replace me")

For 1, look at the vertical Sobel kernel and think about which direction its
positive and negative weights are arranged along. Then rotate that arrangement 45
degrees.

For 2, do question 3 first if you like — but really, just think: if the first
pass finds *edges*, what does a thing that finds edges find when you point it at
a picture made of edges?

In [ ]:
diag = np.array([[-2, -1, 0], [-1, 0, 1], [0, 1, 2]], float)
img = images[0]

fig, ax = plt.subplots(1, 3, figsize=(7, 2.4))
for a, (n, v) in zip(ax, [("original", img),
                          ("diagonal", convolve2d(img, diag)),
                          ("sobel twice", convolve2d(convolve2d(img, vertical), vertical))]):
    a.imshow(v, cmap="gray"); a.set_title(n, fontsize=9); a.axis("off")
plt.tight_layout()

size = 32
for label, k, s, p in [("conv 5x5 no pad", 5, 1, 0), ("maxpool 2", 2, 2, 0),
                       ("conv 3x3 pad 1", 3, 1, 1), ("maxpool 2", 2, 2, 0)]:
    size = conv_out(size, k, s, p)
    print(f"{label:18s} -> {size}x{size}")

Question 2 is the one I'd love you to sit with, because it's the whole chapter in
miniature.

Applying an edge detector to an edge map gives you a **second derivative**. It
responds to places where the edge *strength itself* is changing — which means
corners and line ends, not edges.

You just built a two-layer network by hand. And that is the entire intuition for
depth in a CNN:

- Layer 1 finds edges.
- Layer 2 combines edges into corners and textures.
- Layer 3 combines those into object parts — an eye, a wheel, a doorway.
- Layer 4 combines parts into objects.

Nobody designs that hierarchy. Nobody writes down "layer 3 should find eyes." It
*emerges*, from gradient descent, because that happens to be an efficient way to
turn pixels into labels.

And you can go and look at it. Visualising what maximally activates each filter
is a well-established technique, and the pictures are honestly beautiful — early
layers show edges and colour blobs, middle layers show textures and patterns,
late layers show recognisable eyes and faces and text. It's one of the few places
where you can directly see what a network decided to care about.

Tomorrow: what to do when your input isn't pixels but *categories* — the idea that
turns "user 84,113" into something a model can actually reason about.